# Project 2 — Transaction Intelligence Pipeline
## Layer 1: Data Audit
**Purpose:** Understand raw data quality before cleaning  
**Data:** transactions_raw.csv — 1260 rows, 19 columns  
**Date:** 2026-02-27

In [1]:
from datetime import datetime
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType, BooleanType, TimestampType, LongType

In [2]:
spark = SparkSession.builder \
        .appName("SparkProject2") \
        .master("local[*]") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/12 09:01:21 WARN Utils: Your hostname, Ashu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/12 09:01:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 09:01:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Step 1 — Basic Shape, Schema, & Missing Value Analysis

In [3]:
raw_schema = StructType([
    StructField("transaction_id",                   StringType(), True),
    StructField("user_id",                          StringType(), True),
    StructField("account_number",                   StringType(), True),
    StructField("merchant_id",                      StringType(), True),
    StructField("merchant_category",                StringType(), True),
    StructField("transaction_amount",               StringType(), True),
    StructField("currency",                         StringType(), True),
    StructField("transaction_date",                 StringType(), True),
    StructField("payment_channel",                  StringType(), True),
    StructField("transaction_status",               StringType(), True),
    StructField("failure_reason",                   StringType(), True),
    StructField("device_type",                      StringType(), True),
    StructField("location_city",                    StringType(), True),
    StructField("location_country",                 StringType(), True),
    StructField("is_international",                 StringType(), True),
    StructField("bank_branch_code",                 StringType(), True),
    StructField("relationship_manager",             StringType(), True),
    StructField("approval_status",                  StringType(), True),
    StructField("processing_time_ms",               StringType(), True),
])

raw_df = spark.read \
         .option("header", "true") \
         .schema(raw_schema) \
         .csv("/home/ashu/spark-etl-projects/project2-transaction-intelligence/data/transactions_raw.csv")

print(raw_df.head(5))
total_row_count = raw_df.count()
print(f"Total count: {total_row_count}")

expected_nulls = ["failure_reason"]

print("====Null values per column====")
for column in raw_df.columns:
    missing_count = raw_df.filter(
        sf.col(column).isNull() | (sf.col(column) == " ")
    ).count()

    missing_pct = round(missing_count / total_row_count * 100, 2)
    flag = " ← EXPECTED NULL" if column in expected_nulls and missing_count > 0 \
       else " ← PROBLEM" if missing_count > 0 \
       else ""
    print(f"  {column:25s}: {missing_count:4d} missing ({missing_pct}%){flag}")

[Row(transaction_id='TXNSHZXN8BD', user_id='USROQ89V2', account_number='667718320437', merchant_id='MRCUEE4LJ', merchant_category='HEALTHCARE', transaction_amount='43366.98', currency='INR', transaction_date='2026-02-05 15:08:15', payment_channel='UPI', transaction_status='Success', failure_reason=None, device_type='MOBILE', location_city='Bangalore', location_country='India', is_international='False', bank_branch_code='BR0035', relationship_manager='RM020', approval_status='PENDING_REVIEW', processing_time_ms='1818'), Row(transaction_id='TXN6WE7GJ81', user_id='USRDL87XP', account_number='299665013191', merchant_id='MRC4Z1WMJ', merchant_category='EDUCATION', transaction_amount='8755.77', currency='USD', transaction_date='2026/02/11 22:47', payment_channel='IMPS', transaction_status='SUCCESS', failure_reason=None, device_type='WEB', location_city='Singapore', location_country='UAE', is_international='True', bank_branch_code='BR0043', relationship_manager='RM019', approval_status='DECLIN

## Step 3 — Categorical Column Validation

In [4]:
raw_df.groupBy("transaction_status") \
         .agg(sf.count("transaction_status")) \
         .orderBy(sf.col("transaction_status")).show(30, truncate=20) # Has 14 values - Problem

raw_df.groupBy("merchant_category") \
      .agg(sf.count(sf.col("merchant_category"))) \
      .orderBy(sf.col("merchant_category")).show(20, truncate=True) # Has 14 values - Problem

raw_df.groupBy("payment_channel") \
      .agg(sf.count(sf.col("payment_channel"))) \
      .orderBy(sf.col("payment_channel")).show(20, truncate=True) # Has 5 values only

raw_df.groupBy("device_type") \
      .agg(sf.count(sf.col("device_type"))) \
      .orderBy(sf.col("device_type")).show(20, truncate=20) # Has 4 values only

raw_df.groupBy("currency") \
      .agg(sf.count(sf.col("currency"))) \
      .orderBy(sf.col("currency")).show(20, truncate=20) # Has INR + 5 international

raw_df.groupBy("location_country") \
      .agg(sf.count(sf.col("location_country"))) \
      .orderBy(sf.col("location_country")).show(20, truncate=20) # Has India + 5 international

raw_df.groupBy("is_international") \
      .agg(sf.count(sf.col("is_international"))) \
      .orderBy(sf.col("is_international")).show(20, truncate=True) # Has only two values

raw_df.groupBy("approval_status") \
      .agg(sf.count(sf.col("approval_status"))) \
      .orderBy(sf.col("approval_status")).show(20, truncate=20) # Has only 3 values

+------------------+-------------------------+
|transaction_status|count(transaction_status)|
+------------------+-------------------------+
|            FAILED|                       58|
|            Failed|                       74|
|           PENDING|                       38|
|           Pending|                       53|
|          REVERSED|                       18|
|          Reversed|                       15|
|           SUCCESS|                      306|
|           Success|                      295|
|            failed|                       48|
|           pending|                       38|
|          reversed|                       22|
|           success|                      295|
+------------------+-------------------------+

+-----------------+------------------------+
|merchant_category|count(merchant_category)|
+-----------------+------------------------+
|         CLOTHING|                     116|
|        EDUCATION|                     138|
|      ELECTRONICS|   

## Step 4 — Numeric Sanity Check

In [5]:
clean_transaction_amount = sf.regexp_replace(sf.col("transaction_amount"), ",", ".").cast("double")
raw_df.select(
        sf.min(clean_transaction_amount).alias("min_transaction_amount"),
        sf.max(clean_transaction_amount).alias("max_transaction_amount"),
        sf.round(sf.avg(clean_transaction_amount), 2).alias("avg_transaction_amount"),
        sf.sum(sf.when(clean_transaction_amount < 0, 1).otherwise(0)).alias("negative_count")
        ).show()

raw_df.select(
    sf.min(sf.col("processing_time_ms")).alias("min_processing_time"),
    sf.max(sf.col("processing_time_ms")).alias("max_processing_time"),
    sf.sum(sf.when(sf.col("processing_time_ms") < 0, 1).otherwise(0)).alias("negatinve_processing_time_count")
).show()

+----------------------+----------------------+----------------------+--------------+
|min_transaction_amount|max_transaction_amount|avg_transaction_amount|negative_count|
+----------------------+----------------------+----------------------+--------------+
|                100.29|             496125.65|               46231.0|             0|
+----------------------+----------------------+----------------------+--------------+

+-------------------+-------------------+-------------------------------+
|min_processing_time|max_processing_time|negatinve_processing_time_count|
+-------------------+-------------------+-------------------------------+
|               -124|                998|                             30|
+-------------------+-------------------+-------------------------------+



## Step 5 — Date Format Analysis

In [6]:
print("==== TIMESTAMP FORMAT SAMPLES ====")
raw_df.select("transaction_date") \
      .distinct() \
      .limit(10) \
      .show()

today = datetime.now().strftime("%Y-%m-%d")

# This is tricky on raw string dates
# Simple approach — check for year/month patterns
future_date_count = raw_df.filter(
    sf.col("transaction_date").contains("2026-03") |
    sf.col("transaction_date").contains("2026-04") |
    sf.col("transaction_date").contains("2026-05")
).count()

print(f"Future date counts: {future_date_count}")

==== TIMESTAMP FORMAT SAMPLES ====
+-------------------+
|   transaction_date|
+-------------------+
|2026-02-08 04:56:35|
|   2026/01/11 15:59|
|02/12/2026 04:14:06|
|01/27/2026 14:45:38|
|   2026/02/16 06:59|
|   02-02-2026 06:09|
|2026-04-03 09:04:06|
|   14-02-2026 16:33|
|   22-02-2026 20:36|
|01/25/2026 13:11:43|
+-------------------+

Future date counts: 8


## Step 6 — Duplicate Detection

In [7]:
unique_count = raw_df.distinct().count()
print(f"Unique rows: {unique_count}")
duplicate_count = total_row_count - unique_count
print(f"Duplicate rows: {duplicate_count}")

Unique rows: 1200
Duplicate rows: 60


## Step 7 — Business Logic Validation

In [8]:
failed_count = raw_df.filter(
    (sf.upper(sf.col("transaction_status")) == "FAILED") & (sf.col("failure_reason") == "")
).count()

success_count = raw_df.filter(
    (sf.upper(sf.col("transaction_status")) == "SUCCESS") & (sf.col("failure_reason") != "")
).count()

international_currency_count = raw_df.filter(
    (sf.col("is_international") == "True") & (sf.col("currency") == "INR")
).count()

domestic_foreign_currency_count = raw_df.filter(
    (sf.col("is_international") == "False") & (sf.col("currency").isin("AED", "EUR", "GBP", "SGD", "USD"))
).count()

international_country_count = raw_df.filter(
    (sf.col("is_international") == "False") & (sf.col("location_country") != "India")
).count()

print(f"Failed transactions with empty failure reason: {failed_count}")
print(f"Successful transactions with non-empty failure reason: {success_count}")
print(f"International transactions with non-international currency: {international_currency_count}")
print(f"Domestic transactions with non-India country: {domestic_foreign_currency_count}")
print(f"is_international check with non-India country: {international_country_count}")

Failed transactions with empty failure reason: 0
Successful transactions with non-empty failure reason: 0
International transactions with non-international currency: 0
Domestic transactions with non-India country: 0
is_international check with non-India country: 0


## Audit Summary

In [9]:
missing_amount = raw_df.filter(sf.col("transaction_amount").isNull() | (sf.col("transaction_amount") == " ")).count()
missing_merchant_id = raw_df.filter(sf.col("merchant_id").isNull() | (sf.col("merchant_id") == " ")).count()
negative_process_time = raw_df.filter(sf.col("processing_time_ms") < 0).count()
invalid_merchant_category = raw_df.filter(sf.col("merchant_category").isin("N/A", "PETROL BUNK", "general store", "online")).count()
status_variants = raw_df.select("transaction_status").distinct().count()

print("=" * 55)
print("AUDIT SUMMARY")
print("=" * 55)
print(f"Total rows          : {total_row_count}")
print(f"Duplicates          : {duplicate_count}")
print(f"Missing amount      : {missing_amount}")
print(f"Missing merchant_id : {missing_merchant_id}")
print(f"Negative proc time  : {negative_process_time}")
print(f"Invalid merchant cat: {invalid_merchant_category}")
print(f"Status variants     : {status_variants}")
print(f"Future dates        : {future_date_count}")
print(f"Business violations : 0")
print("=" * 55)

AUDIT SUMMARY
Total rows          : 1260
Duplicates          : 60
Missing amount      : 46
Missing merchant_id : 20
Negative proc time  : 30
Invalid merchant cat: 39
Status variants     : 12
Future dates        : 8
Business violations : 0


## Layer 2 — Data Cleaning

10 cleaning rules applied based on audit findings.

Raw data preserved in raw_df — clean_df is a new DataFrame.

All transformations chained in single assignment for efficiency.

In [10]:
clean_df = (
    raw_df

    # Rule 1: Remove exact duplicates caused by network retry mechanisms
    # Audit found 60 duplicate rows — 5% of total 1260 rows
    .drop_duplicates()

    # Rule 2: Normalize transaction_status to uppercase
    # Audit found 12 variants of 4 actual values (Success/SUCCESS/success etc.)
    .withColumn("transaction_status", sf.upper(sf.col("transaction_status"))) \
    
    # Rule 3: Standardize invalid merchant categories to valid codes
    # PETROL BUNK → FUEL (same category, different naming convention)
    # general store → GROCERY (closest valid category)
    # online, N/A → UNKNOWN (cannot be reliably mapped)
    .withColumn("merchant_category", 
                       sf.when(sf.col("merchant_category")=="PETROL BUNK", "FUEL")
                       .when(sf.col("merchant_category")=="general store", "GROCERY")
                       .when(sf.col("merchant_category")=="online", "UNKNOWN")
                       .when(sf.col("merchant_category")=="N/A", "UNKNOWN").otherwise(sf.col("merchant_category"))
                       ) \
    
    # Rule 4: Normalize transaction_amount
    # CARD channel sends European decimal format (1234,56 instead of 1234.56)
    # Replace comma with dot before casting to Double
    # Empty strings become null automatically on cast — kept for audit trail
    .withColumn("transaction_amount", sf.regexp_replace(sf.col("transaction_amount"), ",", ".").cast("double")) \
    
    # Rule 5: Replace missing merchant_id with traceable placeholder
    # 20 missing merchant_ids found in audit
    # Prefixed with transaction_id to maintain traceability
    # Dropping rows would lose transaction data — kept for compliance
    .withColumn("merchant_id", sf.when((sf.col("merchant_id")=="") | (sf.col("merchant_id").isNull()), sf.concat(sf.lit("UNKNOWN_MERCHANT_"), sf.col("transaction_id"))).otherwise(sf.col("merchant_id"))) \

    # Rule 6: Replace negative processing times with null
    # 30 negative values found in audit — physically impossible values
    # Null is more honest than zero — zero implies instant processing   
    .withColumn("processing_time_ms", sf.when(sf.col("processing_time_ms").cast('int') < 0, None).otherwise(sf.col("processing_time_ms").cast('int'))) \
            
    # Rule 7: Parse transaction_date from 6 different source formats
    # Each payment channel sends dates in different format:
    #   UPI  → yyyy-MM-dd HH:mm:ss
    #   CARD → dd/MM/yyyy HH:mm:ss
    #   RTGS → MM/dd/yyyy HH:mm:ss
    #   IMPS → yyyy/MM/dd HH:mm
    #   NEFT → dd-MM-yyyy HH:mm and MM-dd-yyyy HH:mm
    # coalesce tries each format in order, returns first non-null result       
    .withColumn("transaction_date", sf.coalesce(
                sf.expr("try_to_timestamp(transaction_date, 'yyyy-MM-dd HH:mm:ss')"),
                sf.expr("try_to_timestamp(transaction_date, 'dd/MM/yyyy HH:mm:ss')"),
                sf.expr("try_to_timestamp(transaction_date, 'MM/dd/yyyy HH:mm:ss')"),
                sf.expr("try_to_timestamp(transaction_date, 'yyyy/MM/dd HH:mm')"),
                sf.expr("try_to_timestamp(transaction_date, 'dd-MM-yyyy HH:mm')"),
                sf.expr("try_to_timestamp(transaction_date, 'MM-dd-yyyy HH:mm')")
            )) \
    
    # Rule 8: Flag transaction dates for data quality tracking
    # PARSE_FAILED → none of the 6 formats matched, timestamp is null
    # FUTURE_DATE  → transaction dated beyond today, likely system clock error
    #                kept for compliance audit trail — not dropped
    # OK           → valid parseable date within expected range
    .withColumn("transaction_date_flag", 
                    sf.when(sf.col("transaction_date").isNull(), "PARSE_FAILED")
                    .when(sf.col("transaction_date") > sf.current_date(), "FUTURE_DATE")
                    .otherwise("OK")
                ) \
                
    # Rule 9: Add pipeline audit columns for lineage tracking
    # ingested_at     → exact timestamp this record was processed
    # pipeline_version → version tag for debugging and rollback
    .withColumn("ingested_at", sf.current_timestamp()) \
    .withColumn("pipeline_version", sf.lit("v1.0")) \
            
    # Rule 10: Final schema enforcement
    # is_international cast from String "True"/"False" to Boolean
    # account_number stays String — identifiers should never be numeric
    #   (numeric cast would drop leading zeros and enable meaningless math)
    # processing_time_ms enforced as Integer for correct aggregation behavior
    .withColumn("is_international", sf.col("is_international").cast(BooleanType())) \
    .withColumn("processing_time_ms", sf.col("processing_time_ms").cast(IntegerType()))
)
            

clean_df.printSchema()
clean_df.show()

root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- account_number: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- failure_reason: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_country: string (nullable = true)
 |-- is_international: boolean (nullable = true)
 |-- bank_branch_code: string (nullable = true)
 |-- relationship_manager: string (nullable = true)
 |-- approval_status: string (nullable = true)
 |-- processing_time_ms: integer (nullable = true)
 |-- transaction_date_flag: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)
 |--

## Data Cleaning Summary

In [11]:
print("Row count before cleaning: ", total_row_count)
print("Row count after cleaning: ", clean_df.count())
print("Duplicate row removed after cleaning: ", total_row_count - clean_df.distinct().count())
clean_df.groupBy("transaction_status").count().orderBy("transaction_status").show()
clean_df.groupBy("merchant_category").count().orderBy("merchant_category").show()
clean_df.groupBy("transaction_date_flag").count().show()
clean_df.groupBy("merchant_category").count().orderBy("merchant_category").show()

missing_amount_after_cleaning = clean_df.filter(sf.col("transaction_amount").isNull()).count()
missing_merchant_id_after_cleaning = clean_df.filter(sf.col("merchant_id").isNull() | (sf.col("merchant_id") == " ")).count()
negative_process_time_after_cleaning = clean_df.filter(sf.col("processing_time_ms") < 0).count()
status_variants_after_cleaning = clean_df.select("transaction_status").distinct().count()

print("=" * 55)
print("AUDIT SUMMARY AFTER CLEANING")
print("=" * 55)
print(f"Total rows          : {total_row_count}")
print(f"Duplicates          : {duplicate_count}")
print(f"Missing amount      : {missing_amount_after_cleaning}")
print(f"Missing merchant_id : {missing_merchant_id_after_cleaning}")
print(f"Negative proc time  : {negative_process_time_after_cleaning}")
print(f"Status variants     : {status_variants_after_cleaning}")
print("=" * 55)

Row count before cleaning:  1260
Row count after cleaning:  1200
Duplicate row removed after cleaning:  60
+------------------+-----+
|transaction_status|count|
+------------------+-----+
|            FAILED|  170|
|           PENDING|  124|
|          REVERSED|   51|
|           SUCCESS|  855|
+------------------+-----+

+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|         CLOTHING|  112|
|        EDUCATION|  131|
|      ELECTRONICS|  108|
|    ENTERTAINMENT|  104|
|             FUEL|  118|
|          GROCERY|  150|
|       HEALTHCARE|  113|
|       RESTAURANT|  109|
|           TRAVEL|   99|
|          UNKNOWN|   21|
|        UTILITIES|  135|
+-----------------+-----+

+---------------------+-----+
|transaction_date_flag|count|
+---------------------+-----+
|                   OK| 1102|
|          FUTURE_DATE|   98|
+---------------------+-----+

+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|         CLOTHING|  112|


## Layer 3: Fraud Detection

In [ ]:
merchant_df = spark.read \
              .option("header", "true") \
              .option("inferschema", "true") \
              .csv("/home/ashu/spark-etl-projects/project2-transaction-intelligence/data/merchant_lookup.csv")

# `merchant_df` is a small lookup table containing merchant category metadata
# such as risk levels and transaction limits.  We infer schema so numeric
# fields come in with appropriate types.

# Enrich clean transaction data with merchant info.  A broadcast join
# is safe because the lookup is small and speeds up repeated access later.
enriched_df = clean_df.join(sf.broadcast(merchant_df), "merchant_category")

1200

In [ ]:

# The DataFrame coming into this cell (`enriched_df`) already includes the
# cleaned transaction records.  Before applying any rules we join in the
# merchant lookup table (done in the previous cell) which provides category
# risk levels and maximum allowed transaction amounts.  A broadcast join
# keeps the small lookup table in memory for fast reuse.

# STEP 1 - fraud_flag column
#  - MISSING_AMOUNT: null amounts indicate incomplete data
#  - AMOUNT_LIMIT_EXCEEDED: transaction exceeds merchant's agreed limit
#  - HIGH_VALUE_INTERNATIONAL: large foreign transfer may warrant review
#  - UNKNOWN_MERCHANT_RISK: money to unknown merchant categories
#  - FAILED_HIGH_VALUE: failed payment where a big amount was attempted
#  - CLEAN: none of the above risk conditions matched
enriched_df = enriched_df.withColumn(
    "fraud_flag", 
    sf.when(sf.col("transaction_amount").isNull(), "MISSING_AMOUNT")
    .when(sf.col("transaction_amount") > sf.col("max_single_txn_limit"), "AMOUNT_LIMIT_EXCEEDED")
    .when((sf.col("is_international") == True) & (sf.col("transaction_amount") > 100000), "HIGH_VALUE_INTERNATIONAL")
    .when((sf.col("merchant_category") == "UNKNOWN") & (sf.col("transaction_amount") > 0), "UNKNOWN_MERCHANT_RISK")
    .when((sf.col("transaction_status") == "FAILED") & (sf.col("transaction_amount") > 50000), "FAILED_HIGH_VALUE").otherwise("CLEAN")
)

# display flag counts for sanity checking
enriched_df.groupBy("fraud_flag").count().orderBy("fraud_flag").show()

# STEP 2 - risk_level column
# We escalate non-clean transactions based on the merchant category's
# pre-assigned `category_risk_level`.  This yields four broad buckets
# from LOW up to CRITICAL for the most concerning combinations.
enriched_df = enriched_df.withColumn(
    "risk_level",
    sf.when((sf.col("fraud_flag") != "CLEAN") & (sf.col("category_risk_level") == "HIGH"), "CRITICAL")
    .when((sf.col("fraud_flag") != "CLEAN") & (sf.col("category_risk_level") == "MEDIUM"), "HIGH")
    .when(sf.col("fraud_flag") != "CLEAN", "MEDIUM").otherwise("LOW")
)

# inspect risk level distribution
enriched_df.groupBy("risk_level").count().orderBy("risk_level").show()

# cache the DataFrame for subsequent aggregation steps (multiple passes follow)
enriched_df.cache()
enriched_df.count()


# STEP 3 - MERCHANT RISK PROFILING
# Aggregate by merchant_category to produce metrics useful for monitoring and
# prioritization.  We compute transaction counts, revenue, average amounts,
# failure/reverse rates, international ratios, and how many records were
# flagged by the fraud logic above.  The final `orderBy` sorts categories by
# failure rate so the riskiest segments appear first.
enriched_df.groupBy("merchant_category") \
            .agg(
                sf.count("*").alias("total_transactions"),
                sf.round(sf.sum("transaction_amount"), 2).alias("total_revenue"),
                sf.round(sf.avg("transaction_amount"), 2).alias("avg_transaction_amount"),
                sf.sum(sf.when(sf.col("transaction_status")=="FAILED", 1).otherwise(0)).alias("failure_count"),
                sf.round((sf.sum(sf.when(sf.col("transaction_status")=="FAILED", 1).otherwise(0)) / sf.count("*")), 4).alias("failure_rate"),
                sf.sum(sf.when(sf.col("transaction_status")=="REVERSED", 1).otherwise(0)).alias("reverse_count"),
                sf.round((sf.sum(sf.when(sf.col("transaction_status")=="REVERSED", 1).otherwise(0)) / sf.count("*")), 4).alias("reverse_rate"),
                sf.sum(sf.when(sf.col("is_international")==True, 1).otherwise(0)).alias("international_count"),
                sf.round((sf.sum(sf.when(sf.col("is_international")==True, 1).otherwise(0)) / sf.count("*")), 4).alias("international_ratio"),
                sf.sum(sf.when(sf.col("fraud_flag") != "CLEAN", 1).otherwise(0)).alias("flagged_count")
                ) \
                .select("merchant_category", "total_transactions", "total_revenue", "avg_transaction_amount", "failure_count", "failure_rate", "reverse_count", "reverse_rate", "international_count", "international_ratio", "flagged_count") \
                .orderBy(sf.col("failure_rate").desc()).show()


# STEP 4 - CHANNEL PERFORMANCE ANALYSIS
# A similar set of aggregates but grouped by payment_channel.  Here we care
# about success rate and average processing time to evaluate each channel's
# operational health.
enriched_df.groupBy("payment_channel") \
            .agg(
                sf.count("*").alias("total_transactions"),
                sf.sum(sf.when(sf.col("transaction_status")=="SUCCESS", 1).otherwise(0)).alias("success_count"),
                sf.round((sf.sum(sf.when(sf.col("transaction_status")=="SUCCESS", 1).otherwise(0)) / sf.count("*")), 4).alias("success_rate"),
                sf.round(sf.avg(sf.col("processing_time_ms")), 2).alias("avg_processing_time"),
                sf.round(sf.avg("transaction_amount"), 2).alias("avg_transaction_amount"),
                sf.sum(sf.when(sf.col("is_international")==True, 1).otherwise(0)).alias("international_count")
            ) \
            .select("payment_channel", "total_transactions", "avg_transaction_amount", "success_count", "success_rate", "avg_processing_time", "international_count") \
            .orderBy("success_rate").show()



# Once analysis is complete, unpersist to release the cached data
enriched_df.unpersist()
print(enriched_df.is_cached)


+--------------------+-----+
|          fraud_flag|count|
+--------------------+-----+
|AMOUNT_LIMIT_EXCE...|  190|
|               CLEAN|  940|
|   FAILED_HIGH_VALUE|   10|
|HIGH_VALUE_INTERN...|    8|
|      MISSING_AMOUNT|   44|
|UNKNOWN_MERCHANT_...|    8|
+--------------------+-----+

+----------+-----+
|risk_level|count|
+----------+-----+
|  CRITICAL|   26|
|      HIGH|   73|
|       LOW|  940|
|    MEDIUM|  161|
+----------+-----+

+-----------------+------------------+-------------+----------------------+-------------+------------+-------------+------------+-------------------+-------------------+-------------+
|merchant_category|total_transactions|total_revenue|avg_transaction_amount|failure_count|failure_rate|reverse_count|reverse_rate|international_count|international_ratio|flagged_count|
+-----------------+------------------+-------------+----------------------+-------------+------------+-------------+------------+-------------------+-------------------+-------------+
|   